# Soma de Vetores em CUDA — Benchmark CPU vs GPU

**Disciplina:** Arquitetura de Computadores  
**Autor:** Wésio Filho  
**Professor:** Newarney T. da Costa  

Este notebook implementa a soma de dois vetores (`C[i] = A[i] + B[i]`) em duas versões:
- **CPU**: loop sequencial em C
- **GPU**: kernel CUDA com uma thread por elemento

Em seguida, mede o tempo de cada versão para diferentes tamanhos de N e calcula o speedup.

## Passo 1 — Verificar a GPU

In [ ]:
!nvidia-smi

## Passo 2 — Versão CPU em C

O programa abaixo aloca dois vetores A e B de tamanho N, preenche com valores conhecidos (`A[i] = i`, `B[i] = 2*i`), soma elemento a elemento com um `for` e imprime os primeiros 5 resultados para verificação.

In [ ]:
%%writefile soma_cpu.c
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

int main(int argc, char *argv[]) {
    int N = 1000000;
    if (argc > 1) {
        N = atoi(argv[1]);
        if (N <= 0) { fprintf(stderr, "Erro: N deve ser positivo.\n"); return 1; }
    }

    float *A = (float *)malloc(N * sizeof(float));
    float *B = (float *)malloc(N * sizeof(float));
    float *C = (float *)malloc(N * sizeof(float));

    for (int i = 0; i < N; i++) { A[i] = (float)i; B[i] = 2.0f * (float)i; }

    struct timespec ts_inicio, ts_fim;
    clock_gettime(CLOCK_MONOTONIC, &ts_inicio);

    for (int i = 0; i < N; i++) { C[i] = A[i] + B[i]; }

    clock_gettime(CLOCK_MONOTONIC, &ts_fim);

    double tempo_ms = (ts_fim.tv_sec - ts_inicio.tv_sec) * 1000.0
                    + (ts_fim.tv_nsec - ts_inicio.tv_nsec) / 1000000.0;

    printf("CPU %.6f\n", tempo_ms);

    free(A); free(B); free(C);
    return 0;
}

In [ ]:
!gcc soma_cpu.c -o soma_cpu -lm

In [ ]:
!./soma_cpu 1000000

## Passo 3 — Versão GPU em CUDA

A kernel `somaVetores` faz a mesma soma, mas cada thread da GPU calcula um elemento de forma paralela. O fluxo é:
1. Alocar memória no host (CPU) e no device (GPU)
2. Copiar dados da CPU para a GPU (`cudaMemcpy HostToDevice`)
3. Lançar a kernel com `<<<blocos, threads>>>`
4. Copiar o resultado de volta (`cudaMemcpy DeviceToHost`)
5. Liberar toda a memória

In [ ]:
%%writefile soma_gpu.cu
#include <stdio.h>
#include <stdlib.h>

__global__ void somaVetores(float *A, float *B, float *C, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) {
        C[i] = A[i] + B[i];
    }
}

int main(int argc, char *argv[]) {
    int N = 1000000;
    if (argc > 1) {
        N = atoi(argv[1]);
        if (N <= 0) { fprintf(stderr, "Erro: N deve ser positivo.\n"); return 1; }
    }

    float *h_A = (float *)malloc(N * sizeof(float));
    float *h_B = (float *)malloc(N * sizeof(float));
    float *h_C = (float *)malloc(N * sizeof(float));

    for (int i = 0; i < N; i++) { h_A[i] = (float)i; h_B[i] = 2.0f * (float)i; }

    float *d_A, *d_B, *d_C;
    cudaMalloc((void **)&d_A, N * sizeof(float));
    cudaMalloc((void **)&d_B, N * sizeof(float));
    cudaMalloc((void **)&d_C, N * sizeof(float));

    cudaMemcpy(d_A, h_A, N * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, N * sizeof(float), cudaMemcpyHostToDevice);

    int threadsPorBloco = 256;
    int blocos = (N + threadsPorBloco - 1) / threadsPorBloco;

    cudaEvent_t inicio, fim;
    cudaEventCreate(&inicio);
    cudaEventCreate(&fim);

    /* Medir tempo INCLUINDO transferência de dados (justo) */
    cudaEventRecord(inicio);

    cudaMemcpy(d_A, h_A, N * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, N * sizeof(float), cudaMemcpyHostToDevice);

    somaVetores<<<blocos, threadsPorBloco>>>(d_A, d_B, d_C, N);

    cudaMemcpy(h_C, d_C, N * sizeof(float), cudaMemcpyDeviceToHost);

    cudaEventRecord(fim);
    cudaEventSynchronize(fim);

    float tempoMs = 0.0f;
    cudaEventElapsedTime(&tempoMs, inicio, fim);

    printf("GPU %.6f\n", tempoMs);

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C);
    cudaEventDestroy(inicio); cudaEventDestroy(fim);
    return 0;
}

In [ ]:
!nvcc soma_gpu.cu -o soma_gpu

In [ ]:
!./soma_gpu 1000000

## Passo 4 — Verificar que os resultados são iguais

Ambos devem imprimir os mesmos 5 primeiros valores: C[0]=0, C[1]=3, C[2]=6, C[3]=9, C[4]=12

In [ ]:
%%writefile verifica_cpu.c
#include <stdio.h>
#include <stdlib.h>

int main() {
    int N = 10;
    float A[10], B[10], C[10];
    for (int i = 0; i < N; i++) { A[i] = (float)i; B[i] = 2.0f * (float)i; C[i] = A[i] + B[i]; }
    printf("CPU: ");
    for (int i = 0; i < 5; i++) printf("C[%d]=%.0f ", i, C[i]);
    printf("\n");
    return 0;
}

In [ ]:
!gcc verifica_cpu.c -o verifica_cpu && !./verifica_cpu

In [ ]:
%%writefile verifica_gpu.cu
#include <stdio.h>
#include <stdlib.h>

__global__ void somaVetores(float *A, float *B, float *C, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) C[i] = A[i] + B[i];
}

int main() {
    int N = 10;
    float h_A[10], h_B[10], h_C[10];
    for (int i = 0; i < N; i++) { h_A[i] = (float)i; h_B[i] = 2.0f * (float)i; }
    float *d_A, *d_B, *d_C;
    cudaMalloc((void **)&d_A, N * sizeof(float));
    cudaMalloc((void **)&d_B, N * sizeof(float));
    cudaMalloc((void **)&d_C, N * sizeof(float));
    cudaMemcpy(d_A, h_A, N * sizeof(float), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, N * sizeof(float), cudaMemcpyHostToDevice);
    somaVetores<<<1, 256>>>(d_A, d_B, d_C, N);
    cudaMemcpy(h_C, d_C, N * sizeof(float), cudaMemcpyDeviceToHost);
    printf("GPU: ");
    for (int i = 0; i < 5; i++) printf("C[%d]=%.0f ", i, h_C[i]);
    printf("\n");
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    return 0;
}

In [ ]:
!nvcc verifica_gpu.cu -o verifica_gpu && !./verifica_gpu

## Passo 5 — Benchmark: medir tempo para diferentes tamanhos de N

Rodamos cada versão 5 vezes para cada tamanho de N e calculamos a média.

In [ ]:
import subprocess
import pandas as pd
import re

# Tamanhos de N a testar
tamanhos_N = [1000, 100000, 1000000, 10000000]
num_runs = 5

resultados = []

for N in tamanhos_N:
    for run in range(1, num_runs + 1):
        # CPU
        saida_cpu = subprocess.run(['./soma_cpu', str(N)], capture_output=True, text=True)
        tempo_cpu = float(saida_cpu.stdout.strip().split()[-1])

        # GPU
        saida_gpu = subprocess.run(['./soma_gpu', str(N)], capture_output=True, text=True)
        tempo_gpu = float(saida_gpu.stdout.strip().split()[-1])

        resultados.append({
            'N': N,
            'run': run,
            'tempo_cpu_ms': tempo_cpu,
            'tempo_gpu_ms': tempo_gpu
        })
        print(f'N={N:>10,} | run={run} | CPU={tempo_cpu:.4f} ms | GPU={tempo_gpu:.4f} ms')

# Salvar CSV
df = pd.DataFrame(resultados)
df.to_csv('tempos.csv', index=False)
print('\nResultados salvos em tempos.csv')

## Passo 6 — Calcular médias e speedup

In [ ]:
df = pd.read_csv('tempos.csv')

# Médias por N
medias = df.groupby('N').agg({
    'tempo_cpu_ms': 'mean',
    'tempo_gpu_ms': 'mean'
}).reset_index()

medias['speedup'] = medias['tempo_cpu_ms'] / medias['tempo_gpu_ms']

print('=== Resultados Médios ===')
print(medias.to_string(index=False))

## Passo 7 — Gráficos

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Estilo
plt.style.use('seaborn-v0_8-whitegrid')

# ============================================
# Gráfico 1: Tempo CPU vs GPU em função de N
# ============================================
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(medias['N'], medias['tempo_cpu_ms'], 'o-', label='CPU (sequencial)', color='#e74c3c', linewidth=2, markersize=8)
ax.plot(medias['N'], medias['tempo_gpu_ms'], 's-', label='GPU (CUDA paralelo)', color='#2ecc71', linewidth=2, markersize=8)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Tamanho do vetor (N)', fontsize=12)
ax.set_ylabel('Tempo (ms)', fontsize=12)
ax.set_title('Tempo de Execução: CPU vs GPU', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.set_xticks(tamanhos_N)
ax.set_xticklabels([f'{n:,}' for n in tamanhos_N])

plt.tight_layout()
plt.savefig('grafico_tempos.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo: grafico_tempos.png')

In [ ]:
# ============================================
# Gráfico 2: Speedup em função de N
# ============================================
fig, ax = plt.subplots(figsize=(8, 5))

colors = ['#e74c3c' if s < 1 else '#2ecc71' for s in medias['speedup']]
barras = ax.bar(medias['N'].astype(str), medias['speedup'], color=colors, width=0.6, edgecolor='white', linewidth=1.5)

# Linha de referência speedup = 1
ax.axhline(y=1, color='gray', linestyle='--', linewidth=1, label='Speedup = 1 (empate)')

# Valores sobre as barras
for barra, speedup in zip(barras, medias['speedup']):
    ax.text(barra.get_x() + barra.get_width()/2, barra.get_height() + 0.3,
            f'{speedup:.2f}x', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_xlabel('Tamanho do vetor (N)', fontsize=12)
ax.set_ylabel('Speedup (CPU/GPU)', fontsize=12)
ax.set_title('Speedup: GPU em relação à CPU', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('grafico_speedup.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico salvo: grafico_speedup.png')

## Passo 8 — Copiar resultados para a pasta do repositório

Se estiver rodando no Colab com o repo clonado, copie os arquivos gerados para as pastas certas.

In [ ]:
import shutil
import os

# Criar pasta de resultados se não existir
os.makedirs('resultados', exist_ok=True)

# Copiar arquivos
for f in ['tempos.csv', 'grafico_tempos.png', 'grafico_speedup.png']:
    if os.path.exists(f):
        shutil.copy2(f, f'resultados/{f}')
        print(f'✓ {f} → resultados/{f}')
    else:
        print(f'✗ {f} não encontrado')

---

## Conclusão

- Para **N pequeno** (ex: 1.000), a CPU é mais rápida porque o overhead de transferência de dados entre host e device domina o tempo de execução.
- Para **N grande** (ex: 10.000.000), a GPU é significativamente mais rápida graças ao paralelismo massivo — milhares de threads executam a soma simultaneamente.
- O speedup cresce com N, confirmando que a arquitetura GPU é vantajosa para problemas com grande paralelismo de dados.